# Cross-validate Claude labels with Qwen3-Next-80B-A3B-Thinking

Runs `src/annotation/agent_qwen.py` on a Colab A100 (40GB or 80GB) with int4
quantization. Output: `data/annotations/llm_labels_qwen.jsonl` matching the
Claude JSONL schema, so `validate_agreement.py` can compute Cohen's kappa.

**Expected runtime**: ~437 articles × ~20-40 s each ≈ 2-5 hours on A100 40GB.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Sync repo. Defensive: validates Drive path before linking, cleans stale
# symlinks from prior runs, prints what's in Drive if the path is wrong.
import os, subprocess
REPO_URL    = ''                                  # leave '' to use the Drive copy
DRIVE_REPO  = '/content/drive/MyDrive/CSS2'       # adjust to your Drive folder name
WORK        = '/content/CSS2'

if REPO_URL:
    if not os.path.exists(WORK):
        subprocess.check_call(['git', 'clone', REPO_URL, WORK])
else:
    assert os.path.isdir(DRIVE_REPO), (
        f'DRIVE_REPO not found: {DRIVE_REPO}\n'
        f'Top of Drive: {os.listdir("/content/drive/MyDrive/")[:20]}'
    )
    if os.path.lexists(WORK):
        if os.path.islink(WORK):
            os.unlink(WORK)                       # stale symlink from prior run
        else:
            raise SystemExit(f'{WORK} exists and is not a symlink; remove it manually')
    os.symlink(DRIVE_REPO, WORK)

os.chdir(WORK)
print('cwd =', os.getcwd())
print('contents:', os.listdir('.')[:12])

In [ ]:
# 3. Install deps. Qwen3-Next needs a recent transformers and bitsandbytes for int4.
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate bitsandbytes>=0.43 sentencepiece

In [ ]:
# 4. Cache HF weights on Drive so re-runs don't re-download (~40GB int4 ckpt).
%env HF_HOME=/content/drive/MyDrive/hf_cache
%env TRANSFORMERS_CACHE=/content/drive/MyDrive/hf_cache
# Optional: set HF token if needed for download
# %env HF_TOKEN=hf_...

In [ ]:
# 5. Smoke test: label 5 rubaltic articles first to confirm the pipeline.
!python -m src.annotation.agent_qwen \
    --input data/raw/scraped.jsonl \
    --source rubaltic_lt \
    --quantization int4 \
    --limit 5 \
    --out data/annotations/llm_labels_qwen_smoke.jsonl

In [ ]:
# 6. Full run on all 437 rubaltic articles. --resume lets a crashed run continue.
!python -m src.annotation.agent_qwen \
    --input data/raw/scraped.jsonl \
    --source rubaltic_lt \
    --quantization int4 \
    --resume \
    --out data/annotations/llm_labels_qwen.jsonl

In [ ]:
# 7. Compute Cohen's kappa between Claude (llm_labels.jsonl) and Qwen.
# Treat Qwen as 'gold' for the purpose of comparison — kappa is symmetric.
!python -m src.annotation.validate_agreement \
    --llm  data/annotations/llm_labels.jsonl \
    --gold data/annotations/llm_labels_qwen.jsonl \
    --threshold 0.6 \
    --report results/tables/agreement_claude_vs_qwen.json

In [ ]:
# 8. Inspect disagreement: which articles do Claude and Qwen call differently?
import pandas as pd, json
from pathlib import Path
claude = {r['id']: r for r in [json.loads(l) for l in open('data/annotations/llm_labels.jsonl', encoding='utf-8')]}
qwen   = {r['id']: r for r in [json.loads(l) for l in open('data/annotations/llm_labels_qwen.jsonl', encoding='utf-8')]}
common = sorted(set(claude) & set(qwen))
rows = []
for aid in common:
    if claude[aid]['label'] != qwen[aid]['label']:
        rows.append({
            'id': aid,
            'claude_label': claude[aid]['label'],
            'claude_conf':  claude[aid].get('confidence'),
            'qwen_label':   qwen[aid]['label'],
            'qwen_conf':    qwen[aid].get('confidence'),
            'claude_rationale': claude[aid].get('rationale','')[:200],
            'qwen_rationale':   qwen[aid].get('rationale','')[:200],
        })
df = pd.DataFrame(rows)
print(f'Disagreements: {len(df)} / {len(common)} common')
print(df.groupby(['claude_label','qwen_label']).size().to_string())
df.to_csv('results/tables/disagreements_claude_vs_qwen.csv', index=False)
df.head(20)